# Checkpoints and restart

A rolling checkpoint atomically stores recipe position, partial
operation state, device geometry, active topology, refinement history,
cell bounds, and active targets. Scratch contact buffers are rebuilt on
resume. Cross-revision compatibility is not guaranteed: archive the
generating code revision, settings, backend, and an independent geometry
export. Use a new output path when branching a cleanup study.

In [ ]:
# Path objects are accepted anywhere TANGLE expects a file path.
from pathlib import Path
import tangle

## Every `CheckpointSettings` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `case_id` | Identity used to reject an incompatible restart. | nonempty string |
| `path` | Rolling checkpoint output path. | filesystem path |
| `interval_iterations` | Sparse save cadence. | positive iteration count |
| `resume` | Loads a checkpoint before executing. | boolean |
| `resume_path` | Optional source path distinct from the new output path. | path or `None` |
| `resume_case_id` | Optional expected identity for the source checkpoint. | string or `None` |
| `fresh_formation_on_resume` | Keeps geometry/history but restarts recipe operations. | boolean |

In [ ]:
# case_id prevents accidentally resuming an unrelated recipe that used
# the same filesystem location.
checkpoint = tangle.CheckpointSettings(
    "twenty-ply-needled-v1",
    Path("output/twenty_ply.restart"),
    interval_iterations=500,
    resume=False,
)
fields = [
    "case_id", "path", "interval_iterations", "resume",
    "resume_path", "resume_case_id", "fresh_formation_on_resume",
]
{name: getattr(checkpoint, name) for name in fields}

Set `resume=True` to continue the same rolling file. To branch a saved
state into a new experiment, use a new `path` and point `resume_path`
at the old file. `resume_case_id` validates the source identity.
`fresh_formation_on_resume=True` retains geometry/history but starts the
supplied recipe from operation zero.

In [ ]:
# Branching reads the old checkpoint but writes future progress to a new
# rolling file, leaving the source restart untouched.
branch = checkpoint.replace(
    path=Path("output/alternate_cleanup.restart"),
    resume=True,
    resume_path=checkpoint.path,
    resume_case_id=checkpoint.case_id,
    fresh_formation_on_resume=True,
)
{name: getattr(branch, name) for name in fields}

The complete round trip below first writes a checkpoint from a small
two-fiber recipe, then branches a cleanup from it. A checkpoint is
written only when a run crosses `interval_iterations`, so a short run
with the 500-iteration cadence above would leave nothing to resume.

In [ ]:
from tangle.units import mm, um

# Execution is opt-in in this reference notebook. The default backend
# is "wgpu"; add backend="cpu" on a machine without a supported GPU.
RUN_RECIPE = False
if RUN_RECIPE:
    fiber = tangle.Material("fiber", diameter=20 * um)
    crossing = tangle.FiberCollection("crossing")
    crossing.add_fiber([[0.2 * mm, 0.5 * mm, 0.5 * mm], [0.8 * mm, 0.5 * mm, 0.5 * mm]], fiber)
    crossing.add_fiber([[0.5 * mm, 0.2 * mm, 0.5 * mm], [0.5 * mm, 0.8 * mm, 0.5 * mm]], fiber)
    cell = tangle.Cell([1 * mm, 1 * mm, 1 * mm])
    settings = tangle.RelaxationSettings()

    # Save every 100 iterations so this 300-iteration run leaves a restart.
    source = tangle.CheckpointSettings(
        "two-fiber-demo", Path("output/two_fiber.restart"), interval_iterations=100
    )
    formation = tangle.Recipe(cell)
    formation.insert(crossing)
    formation.relax_for(300)
    formed = formation.run(settings, checkpoint=source)
    print(formed.checkpoint_saves, formed.last_checkpoint_iteration)

    # The branch carries the saved geometry into a new recipe; with
    # fresh_formation_on_resume it runs from that recipe's first operation.
    cleanup = tangle.Recipe(cell)
    cleanup.relax_until_converged()
    result = cleanup.run(
        settings,
        checkpoint=source.replace(
            path=Path("output/two_fiber_cleanup.restart"),
            resume=True,
            resume_path=source.path,
            resume_case_id=source.case_id,
            fresh_formation_on_resume=True,
        ),
    )
    # The resumed, relaxed geometry is a new Assembly on the result.
    print(result.resumed, result.resumed_iteration, result.assembly.fiber_count)